# 03 — Analytics Mart Design

## Purpose and process

This notebook documents the analytical definitions implemented in `dw/marts`.

**Logic chain:** define business question → set table grain → define measures → document SQL source → state validation rules and limitations.


## 1. Mart layer overview

The mart layer converts event-level and order-level warehouse facts into small, reusable tables for descriptive analysis. Three marts are included:

- `marts.fact_daily_funnel`
- `marts.fact_segment_funnel`
- `marts.fact_ltv`

These are local portfolio outputs built from synthetic data, not production reporting tables.


## 2. Daily funnel mart

### Purpose and logic

The daily mart describes how many distinct users reached each funnel stage on each observed date.

**Logic chain:** date → distinct view users → distinct click users → distinct paid-order users → conversion ratios.

### Grain

One row per observed calendar date.

### Measures

| Field | Definition |
| --- | --- |
| `view_users` | Distinct users with a `view` event on the date |
| `click_users` | Distinct users with a `click` event on the date |
| `purchase_users` | Distinct users with a paid order on the date |
| `view_to_click_rate` | `click_users / view_users` |
| `click_to_purchase_rate` | `purchase_users / click_users` |
| `overall_conversion_rate` | `purchase_users / view_users` |

SQL source: `dw/marts/fact_daily_funnel.sql`.


## 3. Segment funnel mart

### Purpose and logic

The segment mart supports descriptive comparison across region and age-group combinations.

**Logic chain:** user segment → distinct stage users → segment conversion ratios.

### Grain

One row per `region × age_group` combination represented in the user dimension.

The same measure definitions used by the daily funnel are applied at segment grain.

SQL source: `dw/marts/fact_segment_funnel.sql`.


## 4. Observed-period revenue per user (LTV proxy)

### Purpose and logic

The LTV mart compares paid revenue per registered user across cohorts and segments.

**Logic chain:** registered user cohort → observed paid revenue per user → cohort-segment aggregation → revenue-per-user proxy.

### Grain

One row per `registration_month × region × age_group` combination.

### Measures

| Field | Definition |
| --- | --- |
| `total_users` | All registered users in the cohort-segment, including non-purchasers |
| `total_revenue` | Paid order revenue observed in the seven-day source period |
| `ltv` | `total_revenue / total_users` |

SQL source: `dw/marts/fact_ltv.sql`.

> The column remains named `ltv` for continuity with the analytical exercise. It is an observed-period proxy, not a forecast of true customer lifetime value.


## 5. Validation criteria

### Funnel checks

- One row per declared grain.
- No duplicated dates or segment combinations.
- `view_users ≥ click_users ≥ purchase_users` for the supplied generated data.
- Conversion ratios are null only when the relevant denominator is zero.

### LTV-proxy checks

- One row per cohort-segment combination.
- All registered users are included in the denominator.
- Only paid orders contribute revenue.
- Mart revenue reconciles to paid revenue in `dw.fact_orders`.


## 6. Interpretation limits

- The source data is synthetic and covers seven days.
- Segment differences are descriptive and may be unstable for small groups.
- Correlation does not identify causes of conversion or revenue differences.
- Real LTV estimation would require a longer observation window, retention behaviour, and a forecasting method.
